### Завдання 1.
Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних;

In [1]:
import os
import urllib.request
from datetime import datetime
import pandas as pd
import glob

def download_data(province_id, folder="vhi_data"):
    if not os.path.exists(folder):
        os.makedirs(folder)
        
    existing_files = [f for f in os.listdir(folder) if f.startswith(f"vhi_id_{province_id}_")]
    if existing_files:
        print(f"Дані для області {province_id} вже існують. Пропускаємо...")
        return
    
    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
    
    try:
        now = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"vhi_id_{province_id}_{now}.csv"
        filepath = os.path.join(folder, filename)
        
        urllib.request.urlretrieve(url, filepath)
        print(f"Завантажено: {filename}")
    except Exception as e:
        print(f"Помилка при завантаженні області {province_id}: {e}")

for i in range(1, 28):
    download_data(i)
print("Завантаження завершено")

Завантажено: vhi_id_1_20260313_130627.csv
Завантажено: vhi_id_2_20260313_130631.csv
Завантажено: vhi_id_3_20260313_130632.csv
Завантажено: vhi_id_4_20260313_130632.csv
Завантажено: vhi_id_5_20260313_130633.csv
Завантажено: vhi_id_6_20260313_130634.csv
Завантажено: vhi_id_7_20260313_130635.csv
Завантажено: vhi_id_8_20260313_130636.csv
Завантажено: vhi_id_9_20260313_130637.csv
Завантажено: vhi_id_10_20260313_130638.csv
Завантажено: vhi_id_11_20260313_130639.csv
Завантажено: vhi_id_12_20260313_130640.csv
Завантажено: vhi_id_13_20260313_130641.csv
Завантажено: vhi_id_14_20260313_130642.csv
Завантажено: vhi_id_15_20260313_130643.csv
Завантажено: vhi_id_16_20260313_130644.csv
Завантажено: vhi_id_17_20260313_130645.csv
Завантажено: vhi_id_18_20260313_130646.csv
Завантажено: vhi_id_19_20260313_130647.csv
Завантажено: vhi_id_20_20260313_130648.csv
Завантажено: vhi_id_21_20260313_130649.csv
Завантажено: vhi_id_22_20260313_130650.csv
Завантажено: vhi_id_23_20260313_130652.csv
Завантажено: vhi_id_

### Завдання 2.
Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області

In [7]:
def read_and_clean_data(directory_path='vhi_data'):
    files = glob.glob(os.path.join(directory_path, "*.csv"))
    dataframes = []
    
    headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']
    
    for file in files:
        try:
            province_id = int(file.split('vhi_id_')[1].split('_')[0])
        except (IndexError, ValueError):
            continue
            
        df = pd.read_csv(file, header=1, names=headers + ['Extra'], skipinitialspace=True)
        df = df[headers] 
        
        df['Year'] = df['Year'].astype(str).str.replace(r'<[^>]*>', '', regex=True).str.strip()
        
        df = df[df['Year'].str.isnumeric()]
        
        df['Year'] = df['Year'].astype(int)
        df['Week'] = df['Week'].astype(int)
        df['VHI'] = df['VHI'].astype(float)
        
        df = df.dropna(subset=['VHI'])
        
        df['Old_ID'] = province_id
        
        dataframes.append(df)
    
    if not dataframes:
        print("Попередження: Файли не знайдені або порожні.")
        return pd.DataFrame()
        
    return pd.concat(dataframes, ignore_index=True)

vhi = read_and_clean_data()
print(f"Загальна кількість очищених записів: {len(vhi)}")
display(vhi.head())

,Year,Week,SMN,SMT,VCI,TCI,VHI,Province,Province_ID
0,1982,1,0.059,258.24,51.11,48.78,49.95,Хмельницька,22
1,1982,2,0.063,261.53,55.89,38.20,47.04,Хмельницька,22
2,1982,3,0.063,263.45,57.30,32.69,44.99,Хмельницька,22
3,1982,4,0.061,265.10,53.96,28.62,41.29,Хмельницька,22
4,1982,5,0.058,266.42,46.87,28.57,37.72,Хмельницька,22


Загальна кількість очищених записів: 60372


,Year,Week,SMN,SMT,VCI,TCI,VHI,Old_ID
0,1982,1,0.059,258.24,51.11,48.78,49.95,10
1,1982,2,0.063,261.53,55.89,38.20,47.04,10
2,1982,3,0.063,263.45,57.30,32.69,44.99,10
3,1982,4,0.061,265.10,53.96,28.62,41.29,10
4,1982,5,0.058,266.42,46.87,28.57,37.72,10


### Завдання 3.
Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька). 


In [3]:
def change_province_indices(df):
    noaa_dict = {
        1: 'Черкаська', 2: 'Чернігівська', 3: 'Чернівецька', 4: 'Республіка Крим', 
        5: 'Дніпропетровська', 6: 'Донецька', 7: 'Івано-Франківська', 8: 'Харківська', 
        9: 'Херсонська', 10: 'Хмельницька', 11: 'Кіровоградська', 12: 'Київська', 
        13: 'Львівська', 14: 'Луганська', 15: 'Миколаївська', 16: 'Одеська', 
        17: 'Полтавська', 18: 'Рівненська', 19: 'Сумська', 20: 'Тернопільська', 
        21: 'Закарпатська', 22: 'Вінницька', 23: 'Волинська', 24: 'Запорізька', 
        25: 'Житомирська', 26: 'м. Київ', 27: 'м. Севастополь'
    }
    
    ukr_order = [
        'Вінницька', 'Волинська', 'Дніпропетровська', 'Донецька', 'Житомирська',
        'Закарпатська', 'Запорізька', 'Івано-Франківська', 'Київська', 'Кіровоградська',
        'Луганська', 'Львівська', 'Миколаївська', 'Одеська', 'Полтавська',
        'Республіка Крим', 'Рівненська', 'Сумська', 'Тернопільська', 'Харківська',
        'Херсонська', 'Хмельницька', 'Черкаська', 'Чернівецька', 'Чернігівська',
        'м. Київ', 'м. Севастополь'
    ]

    new_id_dict = {name: i+1 for i, name in enumerate(ukr_order)}
    
    df['Province'] = df['Old_ID'].map(noaa_dict)
    df['Province_ID'] = df['Province'].map(new_id_dict)

    return df.drop(columns=['Old_ID'])

vhi = change_province_indices(vhi)

print("Дані після зміни індексів")
display(vhi[vhi['Province_ID'] == 10].head())

Дані після зміни індексів


,Year,Week,SMN,SMT,VCI,TCI,VHI,Province,Province_ID
2236,1982,1,0.077,262.50,61.42,34.11,47.76,Кіровоградська,10
2237,1982,2,0.084,264.78,67.23,24.00,45.62,Кіровоградська,10
2238,1982,3,0.086,265.88,67.65,20.41,44.03,Кіровоградська,10
2239,1982,4,0.084,266.90,65.13,19.09,42.11,Кіровоградська,10
2240,1982,5,0.078,266.89,55.58,25.02,40.30,Кіровоградська,10


### Завдання 4. Реалізувати процедури для формування вибірок 
#### 4.1. Ряд VHI для області за вказаний рік.

In [4]:
def get_vhi_by_year(df, province_id, year):
    filtered_data = df[(df['Province_ID'] == province_id) & (df['Year'] == year)]
    result = filtered_data[['Week', 'VHI']]

    return result
    
target_id = 5
target_year = 2015

province_name = vhi.loc[vhi['Province_ID'] == target_id, 'Province'].iloc[0]

print(f"Ряд VHI для області '{province_name}' за {target_year} рік:")

vhi_result = get_vhi_by_year(vhi, target_id, target_year)
display(vhi_result.head(10))

Ряд VHI для області 'Житомирська' за 2015 рік:


,Week,VHI
37492,1,46.62
37493,2,49.11
37494,3,49.81
37495,4,49.21
37496,5,48.98
37497,6,48.16
37498,7,47.40
37499,8,46.76
37500,9,46.48
37501,10,45.77


#### 4.2. Ряд VHI за вказаний діапазон років для вказаних областей.

In [5]:
def get_vhi_by_range(df, province_ids, start_year, end_year):
    
    filtered_data = df[(df['Province_ID'].isin(province_ids)) & 
                       (df['Year'] >= start_year) & 
                       (df['Year'] <= end_year)]
    
    result = filtered_data[['Province_ID', 'Province', 'Year', 'Week', 'VHI']]

    return result.sort_values(by=['Province_ID', 'Year', 'Week'])

start_year=2015
end_year=2017
province_ids=[9,12]
print(f"Ряд VHI для вибраних областей за {start_year}-{end_year} роки:")
vhi_range_data = get_vhi_by_range(vhi, province_ids, start_year, end_year)
display(vhi_range_data.head())

Ряд VHI для вибраних областей за 2015-2017 роки:


,Province_ID,Province,Year,Week,VHI
6188,9,Київська,2015,1,51.09
6189,9,Київська,2015,2,50.33
6190,9,Київська,2015,3,48.00
6191,9,Київська,2015,4,49.56
6192,9,Київська,2015,5,49.31


#### 4.3. Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани.

In [6]:
def get_vhi_statistics(df, province_ids, years):
    filtered_data = df[(df['Province_ID'].isin(province_ids)) & 
                       (df['Year'].isin(years))]
    
    vhi_values = filtered_data['VHI']
    
    if vhi_values.empty:
        return "Немає даних для вибраних параметрів."
    
    stats = {
        'Області (ID)': [province_ids],
        'Роки': [years],
        'Мінімум (Min)': [vhi_values.min()],
        'Максимум (Max)': [vhi_values.max()],
        'Середнє (Mean)': [round(vhi_values.mean(), 2)],
        'Медіана (Median)': [vhi_values.median()]
    }
    return pd.DataFrame(stats)

print("Статистика VHI:")
vhi_stats = get_vhi_statistics(vhi, province_ids=[3, 15], years=[2011, 2020])
display(vhi_stats)

Статистика VHI:


,Області (ID),Роки,Мінімум (Min),Максимум (Max),Середнє (Mean),Медіана (Median)
0,"[3, 15]","[2011, 2020]",19.4,68.14,42.0,41.395
